In [3]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
import warnings
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import train_test_split

In [ ]:
df_inner = pd.read_excel('data/train.xlsx')
df = df_inner
df_inner['group'] = 'inner'
df_outer['group'] = 'outer'

In [ ]:
print(df.shape)
print(df.head(3))

In [ ]:
def f1(x):
    if x == 'NCR':
        return 0 
    elif x == 'CR':
        return 1
    else:
        return -1
df.label = df.label.map(f1)

In [ ]:
df.label.value_counts()

In [ ]:
tableFeats = pd.read_csv('data/featName_table.csv').featName.tolist()

In [ ]:
from sklearn.preprocessing import StandardScaler

X_Ori = df[tableFeats].copy()

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_Ori)

X = pd.DataFrame(X_scaled, columns=X_Ori.columns)

print(X)

In [ ]:
y = df.label.values
y

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

log_reg_l1 = LogisticRegression(
    penalty='l1',       
    solver='liblinear',  
    C=1.0,               
    random_state=42
)

log_reg_l1.fit(X, y)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

lambdas = np.logspace(-3, 1, 100)
Cs = 1 / lambdas

X_array = np.array(X)
y_array = np.array(y)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=198)

all_results = []

for lambda_val, C in zip(lambdas, Cs):
    fold_aucs = []

    for train_idx, val_idx in cv.split(X_array, y_array):
        X_tr, X_val = X_array[train_idx], X_array[val_idx]
        y_tr, y_val = y_array[train_idx], y_array[val_idx]

        model = LogisticRegression(
            penalty='l1',
            solver='liblinear',
            C=C,
            random_state=42,
            max_iter=1000
        )

        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_prob)
        fold_aucs.append(auc)

    row = {
        'lambda': lambda_val,
        'C (1/lambda)': C,
        'Mean AUC': np.mean(fold_aucs),
        'AUC Std': np.std(fold_aucs, ddof=1)
    }

    for i, auc in enumerate(fold_aucs):
        row[f'Fold {i+1} AUC'] = auc

    all_results.append(row)

results_df = pd.DataFrame(all_results)
results_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(np.log10(results_df['lambda']), results_df['Mean AUC'], yerr=results_df['AUC Std'], fmt='o', color='red', ecolor='gray', capsize=3)
best_row = results_df.loc[results_df['Mean AUC'].idxmax()]
best_lambda_log = np.log10(best_row['lambda'])
plt.axvline(best_lambda_log, linestyle='--', color='#87CEFA', label=r'$\lambda_{best}$')
plt.xlabel(r'log ($\lambda$)', fontsize=24, weight='bold')
plt.ylabel('AUC', fontsize=24, weight='bold')
plt.xticks(fontsize=24, weight='bold', rotation=45)
plt.yticks(fontsize=24, weight='bold')
plt.legend()
plt.savefig("lasso-1.pdf", format='pdf', bbox_inches='tight', dpi=1200)
plt.tight_layout()
plt.show()

In [ ]:
best_row = results_df.loc[results_df['Mean AUC'].idxmax()]
best_lambda = best_row['lambda']
best_C = 1 / best_lambda
print("best lambda (alpha) =", best_lambda)
best_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=best_C,
    random_state=42,
    max_iter=1000
)
best_model.fit(X_array, y_array)

In [ ]:
coef = pd.Series(best_model.coef_[0], index=tableFeats)

n_selected = (coef != 0).sum()
n_eliminated = (coef == 0).sum()
print(f"Lasso picked {n_selected} variables and eliminated {n_eliminated} others.\n")

selected_features = coef[coef != 0]
print("Selected Features with Coefficients:")
print(selected_features)

In [ ]:
coefs = []
log_lambdas = np.log10(lambdas)

for C in 1 / lambdas:
    model = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=C,
        random_state=42,
        max_iter=1000
    )
    model.fit(X_array, y_array)
    coefs.append(model.coef_[0])

coef_df = pd.DataFrame(coefs, columns=tableFeats, index=log_lambdas)
coef_df

In [ ]:
from matplotlib.font_manager import FontProperties
import matplotlib.cm as cm

plt.figure(figsize=(12, 6))

num_features = coef_df.shape[1]
cmap = cm.get_cmap('tab20b', num_features)  
colors = [cmap(i) for i in range(num_features)]
for i, col in enumerate(coef_df.columns):
    plt.plot(coef_df.index, coef_df[col], label=col, color=colors[i], linewidth=1.5)

best_lambda = results_df.loc[results_df['Mean AUC'].idxmax(), 'lambda']
plt.axvline(np.log10(best_lambda), linestyle='--', color='#87CEFA', linewidth=2, label=r'$\lambda_{best}$')

plt.axhline(0, color='black', linestyle='-', linewidth=1)

plt.xlabel('log($\\lambda$)', fontsize=24, weight='bold')
plt.ylabel('Coefficient', fontsize=24, weight='bold')
plt.xticks(fontsize=24, weight='bold')
plt.yticks(fontsize=24, weight='bold')
plt.grid(False)

bold_font = FontProperties(weight='bold', size=10)

plt.savefig("lasso-2.pdf", format='pdf', bbox_inches='tight', dpi=1200)
plt.tight_layout()
plt.show()